# Januário Leads — Fase 5 · Retomada sem barra de carregamento

**Novo Google Colab, separado da Fase 4.** Ele retoma a importação a partir do banco do Supabase de homologação, sem perder empresas nem repetir cidades já cadastradas.

## Onde o Colab parou

- A última tentativa visível foi **Itapitanga (BA)**, que terminou com erro; o script vai tentá-la primeiro, caso continue pendente.
- Na verificação da homologação: **3.507 municípios**, **2.063 faltantes**, **179.129 empresas**, **106,26 MiB**.
- O erro observado no Supabase foi uma falha no **comando de download do Overture**, não falta de armazenamento. O código de saída 1 sozinho não permite identificar sua causa exata.

## O que mudou

- A barra visual que parecia carregamento foi retirada. O Colab exibirá linhas simples com o município processado e o resultado; o indicador nativo do próprio Colab continua existindo.
- Os logs volumosos do download ficam ocultos para não lotar a tela.
- Erros de download e timeout são tentados uma segunda vez automaticamente.
- Cidades com falha permanecem na fila de recuperação, guardadas no Supabase: geralmente **6 horas** para nova tentativa; sem dados na fonte, **7 dias**.
- Quando 2 cidades do mesmo estado falharem na mesma sessão, o notebook continua por outros estados, em vez de parar todo o Brasil.
- A contagem nacional permanece real: consulta o banco ao começar e incrementa somente municípios que tiveram registros confirmados.
- O teto continua em **250 MiB**, com pausa preventiva em **245 MiB**. O Colab pode encerrar a sessão antes do limite de 10 horas.

## Como executar

1. Não execute Fase 4 e Fase 5 simultaneamente.
2. Execute as células 1, 2, 3 e 4 em ordem.
3. Digite CONTINUAR_BRASIL quando solicitado e informe a **chave secreta do Supabase Staging** no campo oculto.
4. Quando terminar ou parar, os registros confirmados estarão no Supabase; reabrir a Fase 5 retoma a partir deles.

**Importante:** municípios em recuperação NÃO são considerados importados. Não compartilhe sua chave secreta em mensagens nem no GitHub.


In [ ]:
# Célula 1 — instalar ferramentas necessárias
%pip install -q "overturemaps>=2.1,<3" "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.10,<1"
print("Dependências instaladas.")

In [ ]:
# Célula 2 — disponibilizar o importador localmente, sem acessar o GitHub privado.
# Cópia do script do Januário Leads; não contém chave ou segredo.
from pathlib import Path
_IMPORTER = (
    "#!/usr/bin/env python3\n\"\"\"Import Overture Places into private Supabase table, scoped to Brazil's municipalitie"
    "s.\n\nPre-req:\n    pip install overturemaps geopandas shapely pyogrio\n    export SUPABASE_URL='https://<project>"
    ".supabase.co'\n    export SUPABASE_SERVICE_ROLE_KEY='<service role secret>'\n    python scripts/import_overture_"
    "br.py --states MG,SP --database-max-mb 400\n\nThe runner covers all 27 federal units by default. Each state is s"
    "treamed\nseparately; it does not attempt to download and retain a global Overture file.\nIt stops BEFORE reachin"
    "g a configured database threshold, and reports partial\ncoverage honestly. Never run from a browser or publish "
    "the service-role key.\n\"\"\"\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subp"
    "rocess\nimport sys\nimport tempfile\nimport time\nimport unicodedata\nfrom urllib import request, error\nfrom urllib"
    ".parse import urlparse\n\nIBGE_2024_ROOT = (\n    \"https://geoftp.ibge.gov.br/organizacao_do_territorio/\"\n    \"ma"
    "lhas_territoriais/malhas_municipais/municipio_2024\"\n)\n\ndef state_boundaries_url(state):\n    \"\"\"IBGE offers one"
    " ZIP per UF; do not download the 199 MB national file.\"\"\"\n    if state not in STATE_CODES:\n        raise Value"
    "Error(\"Invalid Brazilian state code\")\n    return f\"{IBGE_2024_ROOT}/UFs/{state}/{state}_Municipios_2024.zip\"\n\n"
    "def ensure_state_boundaries(state, source=None):\n    if source:\n        path = Path(source)\n        if not pat"
    "h.is_file():\n            raise RuntimeError(f\"IBGE boundaries file not found: {path}\")\n        return path\n   "
    " path = Path(\"data/ibge/\") / f\"{state}_Municipios_2024.zip\"\n    if not path.is_file():\n        path.parent.mkd"
    "ir(parents=True, exist_ok=True)\n        url = state_boundaries_url(state)\n        print(f\"[{state}] Downloadin"
    "g IBGE municipal boundaries: {url}\", flush=True)\n        temp = path.with_suffix(\".partial\")\n        try:\n    "
    "        request.urlretrieve(url, temp)\n            if temp.stat().st_size < 512:\n                raise Runtime"
    "Error(\"IBGE boundaries download appears empty\")\n            temp.rename(path)\n        finally:\n            tem"
    "p.unlink(missing_ok=True)\n    return path\nSTATE_CODES = (\n    \"AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR"
    " PE PI \"\n    \"RJ RN RS RO RR SC SP SE TO\"\n).split()\nBATCH_SIZE = 150\n\n\ndef normalize(value):\n    \"\"\"Match the "
    "TypeScript search adapter's city/name normalization exactly.\"\"\"\n    letters = \"\".join(\n        c for c in unic"
    "odedata.normalize(\"NFD\", value.casefold())\n        if unicodedata.category(c) != \"Mn\"\n    )\n    return \" \".joi"
    "n(re.sub(r\"[^a-z0-9]+\", \" \", letters).split())\n\n\ndef first(values):\n    return next((v for v in (values or [])"
    " if isinstance(v, str) and v.strip()), None)\n\n\n# A social profile, WhatsApp contact or link shortener is NOT a"
    " verified\n# business website. This preserves truthful \"sem site\" prospecting filters.\nNON_WEBSITE_HOSTS = froz"
    "enset((\n    \"wa.me\", \"whatsapp.com\", \"api.whatsapp.com\", \"instagram.com\",\n    \"facebook.com\", \"fb.me\", \"m.me\","
    " \"linktr.ee\", \"linktree.com\",\n    \"bit.ly\", \"tinyurl.com\", \"t.co\", \"youtube.com\", \"youtu.be\",\n    \"tiktok.com\""
    ", \"x.com\", \"twitter.com\", \"maps.google.com\",\n    \"goo.gl\", \"beacons.ai\",\n))\n\n\ndef is_business_website(value):\n"
    "    if not isinstance(value, str):\n        return False\n    try:\n        url = urlparse(value.strip())\n       "
    " host = (url.hostname or \"\").lower().removeprefix(\"www.\")\n        return (url.scheme in (\"http\", \"https\")\n    "
    "            and bool(host) and \".\" in host\n                and host not in NON_WEBSITE_HOSTS\n                a"
    "nd not any(host.endswith(\".\" + blocked)\n                            for blocked in NON_WEBSITE_HOSTS))\n    exc"
    "ept ValueError:\n        return False\n\n\ndef first_website(values):\n    return first([value for value in (values"
    " or []) if is_business_website(value)])\n\n\ndef select_diverse_pool(records, maximum):\n    \"\"\"Reserve 40% for co"
    "ntactable companies without identified websites.\n\n    Fill unused slots with the best available business conta"
    "cts. Never promise\n    that a missing website is proof a business actually has no website.\n    \"\"\"\n    ranked "
    "= sorted(records, key=lead_quality, reverse=True)\n    no_site = [r for r in ranked if not r.get(\"website\")\n   "
    "            and (r.get(\"phone\") or r.get(\"email\") or r.get(\"instagram\"))]\n    target_no_site = min(len(no_site"
    "), (maximum * 2 + 4) // 5)\n    chosen = no_site[:target_no_site]\n    selected = {r[\"id\"] for r in chosen}\n    "
    "for row in ranked:\n        if len(chosen) >= maximum:\n            break\n        if row[\"id\"] not in selected:\n"
    "            chosen.append(row)\n            selected.add(row[\"id\"])\n    return chosen\n\n\ndef api(base, key, path"
    ", method=\"GET\", payload=None):\n    data = json.dumps(payload, ensure_ascii=False).encode() if payload is not N"
    "one else None\n    # Legacy service_role JWTs use Authorization; the newer sb_secret_ keys\n    # are opaque and"
    " must be sent only as apikey, never as Bearer JWT.\n    if not (key.startswith(\"eyJ\") or key.startswith(\"sb_sec"
    "ret_\")):\n        raise RuntimeError(\"Use somente a chave secreta de STAGING (sb_secret_ ou service_role legada"
    ").\")\n    headers = {\n        \"apikey\": key,\n        \"Content-Type\": \"application/json\",\n        \"Prefer\": \"res"
    "olution=merge-duplicates,return=minimal\",\n    }\n    if key.startswith(\"eyJ\"):\n        headers[\"Authorization\"]"
    " = f\"Bearer {key}\"\n    for attempt in range(5):\n        try:\n            req = request.Request(base + path, me"
    "thod=method, headers=headers, data=data)\n            with request.urlopen(req, timeout=45) as resp:\n          "
    "      body = resp.read()\n                return json.loads(body) if body else None\n        except error.HTTPEr"
    "ror as exc:\n            detail = exc.read(300).decode(\"utf-8\", errors=\"replace\")\n            if exc.code not i"
    "n (429, 500, 502, 503, 504) or attempt == 4:\n                raise RuntimeError(f\"Supabase API {exc.code}: {de"
    "tail}\") from exc\n        except (error.URLError, TimeoutError):\n            if attempt == 4:\n                r"
    "aise\n        time.sleep(min(2 ** attempt, 15))\n    raise RuntimeError(\"Unreachable API retry state\")\n\n\ndef dat"
    "abase_bytes(base, key):\n    result = api(base, key, \"/rest/v1/rpc/overture_storage_bytes\", method=\"POST\", payl"
    "oad={})\n    if not isinstance(result, int):\n        raise RuntimeError(\"Storage guard unavailable: run SQL mig"
    "ration first.\")\n    return result\n\n\ndef status(base, key, state, phase, count=0, message=None):\n    api(base, "
    "key, \"/rest/v1/overture_import_progress?on_conflict=state\", \"POST\", [{\n        \"state\": state, \"status\": phase"
    ",\n        \"imported_rows\": count, \"last_error\": message,\n        \"checked_at\": time.strftime(\"%Y-%m-%dT%H:%M:%"
    "SZ\", time.gmtime()),\n    }])\n\n\ndef municipalities_index(boundaries):\n    import geopandas as gpd\n\n    frame = "
    "gpd.read_file(\"zip://\" + str(boundaries.resolve()))\n    if frame.crs is None:\n        raise RuntimeError(\"The "
    "IBGE municipal boundary file has no CRS.\")\n    frame = frame.to_crs(epsg=4326).reset_index(drop=True)\n    colu"
    "mns = set(frame.columns)\n    if not {\"NM_MUN\", \"SIGLA_UF\"}.issubset(columns):\n        raise RuntimeError(f\"Une"
    "xpected IBGE municipalities columns: {sorted(columns)}\")\n    frame = frame[frame[\"SIGLA_UF\"].isin(STATE_CODES)"
    "].copy().reset_index(drop=True)\n    return frame\n\n\ndef place_record(feature, municipalities, state):\n    from "
    "shapely.geometry import Point\n\n    props = feature.get(\"properties\") or {}\n    coords = (feature.get(\"geometry"
    "\") or {}).get(\"coordinates\") or []\n    if len(coords) != 2:\n        return None\n    lon, lat = coords\n    if n"
    "ot isinstance(lon, (int, float)) or not isinstance(lat, (int, float)):\n        return None\n    name = (props.g"
    "et(\"names\") or {}).get(\"primary\")\n    # GeoJSON exporters may place the Overture GERS id in Feature.id rather\n"
    "    # than properties.id. Preserve either representation.\n    identifier = props.get(\"id\") or feature.get(\"id\""
    ")\n    if not isinstance(name, str) or not name.strip() or not isinstance(identifier, str):\n        return None"
    "\n    if props.get(\"operating_status\") == \"permanently_closed\":\n        return None\n\n    # State bounding boxes"
    " overlap: map point to the true municipality.\n    point = Point(lon, lat)\n    matches = municipalities.sindex."
    "query(point, predicate=\"intersects\")\n    if len(matches) == 0:\n        return None\n    mun = municipalities.il"
    "oc[int(matches[0])]\n    if mun[\"SIGLA_UF\"] != state:\n        return None\n    address = (props.get(\"addresses\")"
    " or [{}])[0]\n    if not isinstance(address, dict):\n        address = {}\n    taxonomy = props.get(\"taxonomy\") o"
    "r {}\n    confidence = props.get(\"confidence\")\n    socials = props.get(\"socials\") or []\n    websites = props.ge"
    "t(\"websites\") or []\n    instagram = first([s for s in (socials + websites)\n                       if isinstanc"
    "e(s, str) and\n                       re.search(r\"https?://(?:www\\.)?instagram\\.com/\", s, re.I)])\n    return {\n"
    "        \"id\": identifier[:100],\n        \"name\": name.strip()[:240],\n        \"name_key\": normalize(name)[:240],"
    "\n        \"category\": str(taxonomy.get(\"primary\") or props.get(\"basic_category\") or \"\")[:120] or None,\n        "
    "\"city\": str(mun[\"NM_MUN\"])[:120],\n        \"city_key\": normalize(str(mun[\"NM_MUN\"]))[:120],\n        \"state\": st"
    "r(mun[\"SIGLA_UF\"]),\n        \"address\": str(address.get(\"freeform\") or \"\")[:400] or None,\n        \"latitude\": l"
    "at, \"longitude\": lon,\n        \"phone\": (first(props.get(\"phones\")) or \"\")[:80] or None,\n        \"email\": (firs"
    "t(props.get(\"emails\")) or \"\")[:200] or None,\n        \"website\": (first_website(websites) or \"\")[:500] or None,"
    "\n        \"instagram\": instagram[:500] if instagram else None,\n        \"confidence\": confidence if isinstance(c"
    "onfidence, (int, float)) else None,\n        \"operating_status\": str(props.get(\"operating_status\") or \"\")[:35] "
    "or None,\n        \"data_release\": os.getenv(\"OVERTURE_DATA_RELEASE\", \"latest\")[:80],\n    }\n\n\ndef lead_quality(r"
    "ow):\n    \"\"\"Prioritize genuine, contactable establishments within the per-city quota.\"\"\"\n    confidence = row."
    "get(\"confidence\")\n    weight = float(confidence) * 5 if isinstance(confidence, (int, float)) else 0\n    return"
    " (weight\n            + 12 * bool(row.get(\"phone\"))\n            + 11 * bool(row.get(\"email\"))\n            + 7 *"
    " bool(row.get(\"website\"))\n            + 3 * bool(row.get(\"instagram\"))\n            + 2 * bool(row.get(\"categor"
    "y\")))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--state"
    "s\", default=\",\".join(STATE_CODES),\n                        help=\"Comma-separated UF codes; default all 27\")\n  "
    "  parser.add_argument(\"--cities\", default=\"\",\n                        help=\"Optional comma-separated cities, r"
    "equires exactly one UF; useful for a small pilot\")\n    parser.add_argument(\"--boundaries\", default=None,\n     "
    "                   help=\"Optional existing ZIP; defaults to one IBGE ZIP per UF\")\n    parser.add_argument(\"--d"
    "atabase-max-mb\", type=int, default=400)\n    parser.add_argument(\"--max-records-per-state\", type=int, default=0"
    ",\n                        help=\"0=no state cap; nonzero limits state rows and marks coverage partial\")\n    par"
    "ser.add_argument(\"--max-records-per-city\", type=int, default=20,\n                        help=\"Per-municipalit"
    "y safety cap, default 20 (0=unlimited)\")\n    parser.add_argument(\"--dry-run\", action=\"store_true\")\n    args = "
    "parser.parse_args()\n    if args.database_max_mb < 30 or args.database_max_mb > 480:\n        parser.error(\"--da"
    "tabase-max-mb must be between 30 and 480 MB\")\n    if args.max_records_per_state < 0:\n        parser.error(\"--m"
    "ax-records-per-state cannot be negative\")\n    if not 1 <= args.max_records_per_city <= 100:\n        parser.err"
    "or(\"--max-records-per-city must be between 1 and 100\")\n    states = [s.strip().upper() for s in args.states.sp"
    "lit(\",\")]\n    if not states or len(set(states)) != len(states) or any(s not in STATE_CODES for s in states):\n "
    "       parser.error(\"--states must contain unique Brazilian UF codes\")\n    cities = [normalize(s.strip()) for "
    "s in args.cities.split(\",\") if s.strip()]\n    if cities and len(states) != 1:\n        parser.error(\"--cities r"
    "equires exactly one UF in --states\")\n\n    base = os.getenv(\"SUPABASE_URL\", \"\").rstrip(\"/\")\n    key = os.getenv"
    "(\"SUPABASE_SERVICE_ROLE_KEY\", \"\")\n    if not args.dry_run and (not base.startswith(\"https://\") or not key):\n  "
    "      parser.error(\"SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY are required\")\n\n    limit_bytes = args.database"
    "_max_mb * 1024 * 1024\n    for state in states:\n        # Individual state archive keeps network and runner-mem"
    "ory use modest.\n        boundaries = ensure_state_boundaries(state, args.boundaries)\n        municipalities = "
    "municipalities_index(boundaries)\n        state_munis = municipalities.loc[municipalities.SIGLA_UF == state].co"
    "py()\n        if cities:\n            state_munis = state_munis.loc[\n                state_munis.NM_MUN.map(lamb"
    "da name: normalize(str(name)) in cities)\n            ].copy()\n            found_cities = {normalize(str(name))"
    " for name in state_munis.NM_MUN}\n            if set(cities) != found_cities:\n                raise RuntimeErro"
    "r(f\"Municipality not found in {state}: {sorted(set(cities) - found_cities)}\")\n        if state_munis.empty:\n  "
    "          raise RuntimeError(f\"IBGE archive does not cover {state}\")\n        west, south, east, north = state_"
    "munis.total_bounds\n        # Slightly wider rectangle avoids border rounding gaps.\n        bbox = \",\".join(f\"{"
    "v:.6f}\" for v in (west - .005, south - .005, east + .005, north + .005))\n        print(f\"[{state}] Overture bb"
    "ox {bbox}\", flush=True)\n        if args.dry_run:\n            continue\n        storage_before = database_bytes("
    "base, key)\n        if storage_before >= limit_bytes:\n            print(f\"STOP: database reached configured {ar"
    "gs.database_max_mb} MB safety limit.\")\n            return 2\n        total = 0\n        status(base, key, state,"
    " \"running\")\n        try:\n            with tempfile.TemporaryDirectory(prefix=\"overture-br-\") as temp:\n        "
    "        out = Path(temp) / f\"{state}.geojsonseq\"\n                subprocess.run([\"overturemaps\", \"download\", f"
    "\"--bbox={bbox}\",\n                                \"--type=place\", \"-f\", \"geojsonseq\", \"-o\", str(out)],\n        "
    "                       check=True)\n                # Store only a small, quality-ranked per-city selection in "
    "memory.\n                city_records = {}\n                city_capped = False\n                with out.open(\"r"
    "\", encoding=\"utf-8\") as src:\n                    for line in src:\n                        line = line.lstrip(\""
    "\\x1e\").strip()\n                        if not line:\n                            continue\n                     "
    "   row = place_record(json.loads(line), state_munis, state)\n                        if row is None:\n          "
    "                  continue\n                        # Keep two bounded quality pools per municipality.\n        "
    "                # Otherwise the top 30 are almost always ALL \"with site\".\n                        groups = cit"
    "y_records.setdefault(row[\"city_key\"], {\n                            \"with_site\": [], \"without_site\": [],\n     "
    "                   })\n                        pool = groups[\"with_site\" if row.get(\"website\") else \"without_si"
    "te\"]\n                        if len(pool) < args.max_records_per_city:\n                            pool.append"
    "(row)\n                        else:\n                            city_capped = True\n                           "
    " worst = min(range(len(pool)),\n                                        key=lambda idx: lead_quality(pool[idx])"
    ")\n                            if lead_quality(row) > lead_quality(pool[worst]):\n                              "
    "  pool[worst] = row\n\n                # Interleave municipality samples for nationwide fair coverage.\n         "
    "       ranked_cities = {\n                    city: select_diverse_pool(groups[\"with_site\"] + groups[\"without_s"
    "ite\"],\n                                              args.max_records_per_city)\n                    for city, "
    "groups in city_records.items()\n                }\n                selected = []\n                for rank in ran"
    "ge(args.max_records_per_city):\n                    for city in sorted(ranked_cities):\n                        "
    "pool = ranked_cities[city]\n                        if rank < len(pool):\n                            selected.a"
    "ppend(pool[rank])\n\n                if not selected:\n                    raise RuntimeError(\"Overture returned "
    "no usable places for this state\")\n                # A municipality-only pilot never marks the whole state as i"
    "mported.\n                partial = city_capped or bool(cities)\n                if args.max_records_per_state a"
    "nd len(selected) > args.max_records_per_state:\n                    selected = selected[:args.max_records_per_s"
    "tate]\n                    partial = True\n                for start in range(0, len(selected), BATCH_SIZE):\n   "
    "                 if database_bytes(base, key) >= limit_bytes:\n                        partial = True\n         "
    "               break\n                    batch = selected[start:start + BATCH_SIZE]\n                    api(ba"
    "se, key, \"/rest/v1/overture_places?on_conflict=id\", \"POST\", batch)\n                    total += len(batch)\n   "
    "             storage_after = database_bytes(base, key)\n                status(base, key, state, \"partial\" if p"
    "artial else \"complete\", total)\n                delta = storage_after - storage_before\n                bytes_pe"
    "r_imported_row = delta // total if total else 0\n                print(f\"[{state}] {total} imported; \"\n        "
    "              f\"database before={storage_before}, after={storage_after}, \"\n                      f\"delta={delt"
    "a}, average={bytes_per_imported_row} B/row; \"\n                      f\"{'PARTIAL' if partial else 'COMPLETE'}\","
    " flush=True)\n                if storage_after >= limit_bytes:\n                    print(\"STOP: database safety"
    " threshold; remaining states NOT imported.\", flush=True)\n                    return 2\n        except Exception"
    " as exc:\n            status(base, key, state, \"error\", total, str(exc)[:200])\n            raise\n    print(\"Ing"
    "estion pass ended. Check overture_import_progress for coverage.\", flush=True)\n    return 0\n\n\nif __name__ == \"_"
    "_main__\":\n    sys.exit(main())\n"
)
_SCRIPT = Path("/content/import_overture_br.py")
compile(_IMPORTER, str(_SCRIPT), "exec")  # valida a sintaxe antes de executar
_SCRIPT.write_text(_IMPORTER, encoding="utf-8")
print("Importador preparado. Compatível com sb_secret_ e service_role legado; nenhum dado enviado ao Supabase.")


In [ ]:
# Célula 3 — RETOMADA NACIONAL. Não usa uma meta antiga fixa de "5.288".
# O número de cidades restantes será calculado EM TEMPO REAL no Supabase.
TOTAL_LOCALIDADES_BRASIL = 5570
MAX_NOVOS_POR_SESSAO = 5570  #@param {type:"integer"}
LIMITE_POR_MUNICIPIO = 60  #@param {type:"integer"}
TEMPO_MAX_MINUTOS = 600  #@param {type:"integer"}
MAX_FALHAS_POR_EXECUCAO = 50
MAX_FALHAS_CONSECUTIVAS = 8
MAX_FALHAS_POR_UF = 2
HORAS_ESPERA_FALHAS = 6
DIAS_ESPERA_SEM_DADOS = 7
# Preferência de retomada do print recebido: se ela ainda não estiver no banco,
# será a primeira tentativa desta sessão.
PONTO_DE_RETOMADA = ("BA", "Itapitanga")

# Cidades priorizadas (sem chamadas externas na seleção). Depois usa os ZIPs
# oficiais da malha IBGE, reutilizando o download já feito pelo importador.
PRIORIDADES = {
    "MG": ["Uberlândia", "Juiz de Fora", "Uberaba", "Montes Claros", "Divinópolis", "Contagem", "Betim", "Sete Lagoas"],
    "SP": ["Sorocaba", "São José dos Campos", "Santos", "Bauru", "Piracicaba", "São José do Rio Preto", "São Carlos", "Franca"],
    "RJ": ["Volta Redonda", "São Gonçalo", "Nova Iguaçu", "Duque de Caxias", "Campos dos Goytacazes", "Macaé", "Cabo Frio"],
    "AC": ["Sena Madureira", "Brasiléia", "Tarauacá", "Feijó"],
    "AL": ["Palmeira dos Índios", "Rio Largo", "Penedo", "União dos Palmares"],
    "AP": ["Laranjal do Jari", "Oiapoque", "Porto Grande"],
    "AM": ["Itacoatiara", "Manacapuru", "Tefé", "Coari", "Tabatinga"],
    "BA": ["Vitória da Conquista", "Camaçari", "Ilhéus", "Itabuna", "Juazeiro", "Barreiras", "Porto Seguro"],
    "CE": ["Sobral", "Maracanaú", "Caucaia", "Crato", "Iguatu"],
    "ES": ["Serra", "Cariacica", "Cachoeiro de Itapemirim", "Linhares", "Colatina"],
    "GO": ["Anápolis", "Rio Verde", "Trindade", "Jataí", "Itumbiara", "Catalão"],
    "MA": ["Caxias", "Timon", "Açailândia", "Bacabal", "Balsas"],
    "MT": ["Rondonópolis", "Sinop", "Sorriso", "Tangará da Serra", "Cáceres"],
    "MS": ["Três Lagoas", "Corumbá", "Ponta Porã", "Naviraí", "Nova Andradina"],
    "PA": ["Santarém", "Marabá", "Castanhal", "Parauapebas", "Altamira"],
    "PB": ["Patos", "Sousa", "Bayeux", "Santa Rita", "Cajazeiras"],
    "PR": ["Maringá", "Cascavel", "Ponta Grossa", "Foz do Iguaçu", "São José dos Pinhais", "Guarapuava"],
    "PE": ["Olinda", "Paulista", "Caruaru", "Petrolina", "Garanhuns"],
    "PI": ["Picos", "Floriano", "Piripiri", "Campo Maior", "Oeiras"],
    "RN": ["Parnamirim", "Caicó", "Macaíba", "Assú", "Ceará-Mirim"],
    "RS": ["Pelotas", "Santa Maria", "Canoas", "Novo Hamburgo", "Passo Fundo", "Bento Gonçalves"],
    "RO": ["Ariquemes", "Vilhena", "Cacoal", "Jaru", "Rolim de Moura"],
    "RR": ["Alto Alegre", "Caracaraí", "Bonfim", "Mucajaí", "Cantá"],
    "SC": ["Blumenau", "Itajaí", "Chapecó", "Criciúma", "São José", "Balneário Camboriú"],
    "SE": ["Itabaiana", "Lagarto", "Estância", "São Cristóvão", "Propriá"],
    "TO": ["Gurupi", "Porto Nacional", "Paraíso do Tocantins", "Colinas do Tocantins"],
}

# O DF tem somente Brasília como município (já importada).
# Revezamento entre UFs para ampliar a presença nacional progressivamente.
ORDEM_UFS = ("MG SP RJ AC AL AP AM BA CE ES GO MA MT MS PA PB PR PE PI RN RS RO RR SC SE TO").split()
assert len(ORDEM_UFS) == 26 and set(ORDEM_UFS) == set(PRIORIDADES)
assert TOTAL_LOCALIDADES_BRASIL == 5570
assert type(MAX_NOVOS_POR_SESSAO) is int and 1 <= MAX_NOVOS_POR_SESSAO <= 5570
assert type(LIMITE_POR_MUNICIPIO) is int and 1 <= LIMITE_POR_MUNICIPIO <= 100
assert type(TEMPO_MAX_MINUTOS) is int and 15 <= TEMPO_MAX_MINUTOS <= 660
print("FASE 5: RETOMADA COM LOG LIMPO — limite de", MAX_NOVOS_POR_SESSAO,
      "novas localidades por sessão; até", LIMITE_POR_MUNICIPIO, "empresas por município.")
print("Contagem nacional atualizada ao iniciar; sem barra de carregamento por cidade.")
print("Ponto inicial preferido:", PONTO_DE_RETOMADA[1] + "/" + PONTO_DE_RETOMADA[0])
print("Proteção de armazenamento: teto 250 MiB; pausa antecipada em 245 MiB.")
print("Após as prioridades, o sistema lê os próximos municípios dos ZIPs da malha IBGE.")
print("Nenhum dado importado nesta célula.")


In [ ]:
# Célula 4 — RETOMADA SEM BARRAS: log curto, tenta novamente, guarda pendências.
# O Supabase é a fonte real para retomar de onde parou.
from getpass import getpass
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError
import json
import os
import re
import subprocess
import sys
import time
import unicodedata

_STAGING = "https://prlrxjothrslhswpkoza.supabase.co"
_MIB = 1024 * 1024
# Limite preventivo ampliado de 110 para 250 MiB.
# Não altera a cota real do Supabase Free (500 MB, aprox. 476,8 MiB).
# Margem de 5 MiB antes do teto para o último lote e crescimento do índice.
_TETO_LIMITE_MIB = 250
_TETO_PREVENTIVO_MIB = _TETO_LIMITE_MIB - 5
_TETO_IMPORTADOR_MIB = _TETO_LIMITE_MIB
_PAGINA = 1000

def _normalizar(v):
    s = "".join(c for c in unicodedata.normalize("NFD", v.casefold())
                if unicodedata.category(c) != "Mn")
    return " ".join(re.sub(r"[^a-z0-9]+", " ", s).split())

def _rest(chave, rota, metodo="GET", corpo=None, contar=False):
    # JWT service_role legado usa Bearer; chave sb_secret_ nova usa só apikey.
    headers = {
        "apikey": chave,
        "Accept": "application/json", "Content-Type": "application/json",
        "Prefer": "count=exact",
    }
    if chave.startswith("eyJ"):
        headers["Authorization"] = "Bearer " + chave
    req = Request(_STAGING + rota, headers=headers, method=metodo,
                  data=json.dumps(corpo).encode("utf-8") if corpo is not None else None)
    with urlopen(req, timeout=45) as resp:
        if contar:
            total = (resp.headers.get("Content-Range") or "").rsplit("/", 1)[-1]
            if not total.isdigit():
                raise RuntimeError("O Supabase não retornou uma contagem verificável.")
            return int(total)
        return json.loads(resp.read())

def _espaco(chave):
    n = _rest(chave, "/rest/v1/rpc/overture_storage_bytes", "POST", {})
    if not isinstance(n, int) or n <= 0:
        raise RuntimeError("Falha ao consultar armazenamento da homologação.")
    return n

def _existentes(chave):
    """Paginação completa: não perde cidades após as primeiras 1000."""
    cidades = set()
    posicao = 0
    while True:
        rota = ("/rest/v1/overture_import_city_progress?select=state,city_key"
                + "&order=state.asc,city_key.asc"
                + "&limit=" + str(_PAGINA) + "&offset=" + str(posicao))
        registros = _rest(chave, rota)
        if not isinstance(registros, list):
            raise RuntimeError("Não foi possível ler a cobertura municipal existente.")
        for r in registros:
            if isinstance(r.get("state"), str) and isinstance(r.get("city_key"), str):
                cidades.add((r["state"], r["city_key"]))
        if len(registros) < _PAGINA:
            return cidades
        posicao += len(registros)

def _municipios_malha(uf, cache):
    """Depois das cidades prioritárias, reutiliza os mesmos ZIPs do IBGE do importador.

    NÃO consulta servicodados.ibge.gov.br: esse endpoint causou TimeoutError
    antes de qualquer empresa ser importada. A malha municipal de 2024 também
    é utilizada pelo importador e fica em cache em data/ibge no Colab.
    """
    if uf in cache:
        return cache[uf]
    if "_helpers" not in cache:
        from runpy import run_path
        cache["_helpers"] = run_path(str(_SCRIPT))
    helpers = cache["_helpers"]
    boundaries = helpers["ensure_state_boundaries"](uf)
    municipios = helpers["municipalities_index"](boundaries)
    nomes = {
        _normalizar(str(row.NM_MUN)): str(row.NM_MUN)
        for row in municipios.itertuples()
        if str(row.SIGLA_UF) == uf and str(row.NM_MUN).strip()
    }
    if not nomes:
        raise RuntimeError("A malha municipal do IBGE não contém municípios válidos em " + uf)
    cache[uf] = nomes
    return nomes

def _selecionar(existentes, cache, ja_tentados, pendentes, ufs_pausadas):
    # Falhas aguardando nova tentativa não são tratadas como importações concluídas.
    bloqueados = ja_tentados | pendentes
    # Primeira tentativa: a última cidade em processamento na captura anterior.
    retomada = (PONTO_DE_RETOMADA[0], _normalizar(PONTO_DE_RETOMADA[1]))
    if (PONTO_DE_RETOMADA[0] not in ufs_pausadas and retomada not in existentes and retomada not in bloqueados):
        return PONTO_DE_RETOMADA[0], PONTO_DE_RETOMADA[1]
    quantos = {uf: 0 for uf in ORDEM_UFS}
    for uf, key in existentes:
        if uf in quantos:
            quantos[uf] += 1
    ordem = sorted(ORDEM_UFS, key=lambda x: (quantos[x], ORDEM_UFS.index(x)))

    # PRIMEIRA PASSAGEM: prioriza municípios conhecidos, sem nenhuma chamada
    # externa. O importador valida os nomes na malha oficial antes de escrever.
    for uf in ordem:
        if uf in ufs_pausadas:
            continue
        for cidade in PRIORIDADES[uf]:
            key = (uf, _normalizar(cidade))
            if key not in existentes and key not in bloqueados:
                return uf, cidade

    # SEGUNDA PASSAGEM: quando todas as prioridades forem usadas, aproveita
    # os arquivos IBGE já baixados pelo importador (ou baixa só o ZIP da UF).
    # Não depende da API de localidades que excedeu o tempo de conexão.
    for uf in ordem:
        if uf in ufs_pausadas:
            continue
        try:
            nomes = _municipios_malha(uf, cache)
        except Exception as exc:
            ufs_pausadas.add(uf)
            print("MALHA DO IBGE INDISPONÍVEL:", uf, type(exc).__name__,
                  "| tentando outras UFs.", flush=True)
            continue
        for key in sorted(nomes):
            if (uf, key) not in existentes and (uf, key) not in bloqueados:
                return uf, nomes[key]
    return None

def _contagem(chave, uf, cidade):
    query = urlencode({
        "select": "id", "state": "eq." + uf,
        "city_key": "eq." + _normalizar(cidade),
    })
    return _rest(chave, "/rest/v1/overture_places?" + query, metodo="HEAD", contar=True)

def _painel_nacional(existentes, qtd_empresas=None, titulo="PROGRESSO NACIONAL"):
    """Sem animação ou barra repetida; imprime somente a contagem."""
    total = len(existentes)
    faltam = max(0, TOTAL_LOCALIDADES_BRASIL - total)
    print(f"{titulo}: {total:,}/{TOTAL_LOCALIDADES_BRASIL:,} municípios "
          f"({total*100/TOTAL_LOCALIDADES_BRASIL:.2f}%) | FALTAM {faltam:,}".replace(",", "."),
          flush=True)
    if qtd_empresas is not None:
        print("Empresas no banco:", f"{qtd_empresas:,}".replace(",", "."))
    return faltam

def _total_empresas(chave):
    return _rest(chave, "/rest/v1/overture_places?select=id",
                 metodo="HEAD", contar=True)

def _carregar_pendencias(chave):
    """Consulta as cidades com falhas temporariamente em espera na homologação."""
    from datetime import datetime, timezone
    agora = datetime.now(timezone.utc).isoformat(timespec="seconds")
    pendentes, offset = set(), 0
    while True:
        qs = urlencode({
            "select": "state,city_key", "retry_after": "gt." + agora,
            "order": "state.asc,city_key.asc",
            "limit": str(_PAGINA), "offset": str(offset),
        })
        itens = _rest(chave, "/rest/v1/overture_auto_city_attempts?" + qs)
        if not isinstance(itens, list):
            raise RuntimeError("Não foi possível consultar pendências.")
        pendentes.update((x["state"], x["city_key"]) for x in itens)
        if len(itens) < _PAGINA:
            return pendentes
        offset += len(itens)

def _registrar_pendencia(chave, uf, cidade, motivo):
    """Persiste adiamento sem armazenar logs, segredos ou dados confidenciais."""
    from datetime import datetime, timedelta, timezone
    prazo = timedelta(days=DIAS_ESPERA_SEM_DADOS) if motivo == "sem_dados_overture" else timedelta(hours=HORAS_ESPERA_FALHAS)
    body = {
        "state": uf, "city": cidade, "city_key": _normalizar(cidade),
        "last_error": motivo[:100],
        "retry_after": (datetime.now(timezone.utc)+prazo).isoformat(),
    }
    headers = {"apikey": chave, "Content-Type": "application/json",
               "Prefer": "resolution=merge-duplicates,return=minimal"}
    if chave.startswith("eyJ"):
        headers["Authorization"] = "Bearer " + chave
    req = Request(_STAGING + "/rest/v1/overture_auto_city_attempts?on_conflict=state,city_key",
                  method="POST", headers=headers, data=json.dumps(body).encode("utf-8"))
    with urlopen(req, timeout=40) as resp:
        resp.read()

def _classificar_falha(saida, codigo):
    texto = (saida or "").lower()
    if "no usable places" in texto or "no usable" in texto:
        return "sem_dados_overture"
    if "database reached" in texto or "database safety" in texto:
        return "teto_do_banco"
    if "unauthorized" in texto or "invalid api key" in texto:
        return "autenticacao"
    if ("overturemaps" in texto or "download" in texto or "connection" in texto
            or "timed out" in texto or "s3" in texto or "duckdb" in texto):
        return "download_overture"
    return "erro_importador_" + str(codigo)

def _executar():
    print("FASE 5 — RETOMADA NACIONAL SEM BARRAS REPETIDAS.")
    print("Teto configurado:", _TETO_LIMITE_MIB, "MiB | pausa preventiva:", _TETO_PREVENTIVO_MIB, "MiB.")
    confirmar = input("Para continuar importando NOVOS municípios só na HOMOLOGAÇÃO, digite CONTINUAR_BRASIL: ").strip().upper()
    if confirmar != "CONTINUAR_BRASIL":
        print("Cancelado; nenhum dado alterado.")
        return
    chave = getpass("Chave SECRETA da HOMOLOGAÇÃO (sb_secret_ ou service_role antiga; oculta): ").strip()
    if not chave:
        print("Chave vazia. Nenhuma importação iniciada.")
        return
    if chave.startswith(("sb_publishable_", "sbp_")):
        print("CHAVE INCORRETA: foi fornecida uma chave pública. Use a chave secreta do projeto STAGING.")
        return
    if not (chave.startswith("sb_secret_") or chave.startswith("eyJ")):
        print("FORMATO DE CHAVE NÃO RECONHECIDO. Use sb_secret_ ou service_role JWT antigo da HOMOLOGAÇÃO.")
        return
    try:
        tamanho = _espaco(chave)
        print("STAGING confirmado:", round(tamanho/_MIB, 2), "MiB usados.")
        if tamanho >= _TETO_PREVENTIVO_MIB * _MIB:
            print("PARADA PREVENTIVA: atingiu", _TETO_PREVENTIVO_MIB, "MiB.")
            return
        existentes = _existentes(chave)
        pendentes = _carregar_pendencias(chave)
        print("Cidades com falhas aguardando outra tentativa:", len(pendentes))
        restantes_inicio = _painel_nacional(
            existentes, qtd_empresas=_total_empresas(chave),
            titulo="INÍCIO DA RETOMADA"
        )
        retomada_chave = (PONTO_DE_RETOMADA[0], _normalizar(PONTO_DE_RETOMADA[1]))
        if retomada_chave in existentes:
            print("Ponto do print já concluído:", PONTO_DE_RETOMADA[1] + "/" + PONTO_DE_RETOMADA[0])
        else:
            print("Primeira tentativa agendada:", PONTO_DE_RETOMADA[1] + "/" + PONTO_DE_RETOMADA[0])
        limite_esta_sessao = min(MAX_NOVOS_POR_SESSAO, restantes_inicio)
        if limite_esta_sessao == 0:
            print("META JÁ ALCANÇADA: nada a importar nesta sessão.")
            return
        cache_ibge = {}
        tentados = set()
        ufs_pausadas = set()
        falhas_por_uf = {}
        confirmados = 0
        falhas = 0
        falhas_consecutivas = 0
        inicio = time.monotonic()
        deadline = inicio + TEMPO_MAX_MINUTOS * 60
        print("META DESTA SESSÃO: até", limite_esta_sessao,
              "NOVAS localidades pendentes, por até", TEMPO_MAX_MINUTOS, "minutos.")
        while confirmados < limite_esta_sessao and len(existentes) < TOTAL_LOCALIDADES_BRASIL:
            restante = deadline - time.monotonic()
            if restante < 360:
                print("PAUSA SEGURA: tempo da sessão chegando ao limite. "
                      "Pode executar esta célula novamente para continuar.")
                break
            if falhas >= MAX_FALHAS_POR_EXECUCAO or falhas_consecutivas >= MAX_FALHAS_CONSECUTIVAS:
                print("PAUSA SEGURA:", falhas, "falhas nesta sessão,",
                      falhas_consecutivas, "seguidas. Investigue os erros antes de retomar.")
                break
            if _espaco(chave) >= _TETO_PREVENTIVO_MIB * _MIB:
                print("PARADA PREVENTIVA: não iniciar novos municípios perto do teto.")
                break
            proximo = _selecionar(existentes, cache_ibge, tentados, pendentes, ufs_pausadas)
            if proximo is None:
                print("NENHUM MUNICÍPIO ELEGÍVEL AGORA: veja as cidades pendentes em recuperação.")
                break
            uf, cidade = proximo
            key = (uf, _normalizar(cidade))
            tentados.add(key)
            antes = _contagem(chave, uf, cidade)
            if antes:
                print("JÁ EXISTE:", cidade + "/" + uf, "|", antes,
                      "empresas. Atualizando a lista e seguindo.")
                existentes.add(key)
                continue
            print("INICIANDO:", cidade + "/" + uf,
                  "| até", LIMITE_POR_MUNICIPIO, "empresas",
                  "| faltam:", max(0, TOTAL_LOCALIDADES_BRASIL - len(existentes)),
                  "| nesta sessão:", confirmados, "/", limite_esta_sessao, flush=True)
            cmd = [
                sys.executable, str(_SCRIPT), "--states", uf, "--cities", cidade,
                "--max-records-per-state", str(LIMITE_POR_MUNICIPIO),
                "--max-records-per-city", str(LIMITE_POR_MUNICIPIO),
                "--database-max-mb", str(_TETO_IMPORTADOR_MIB),
            ]
            ambiente = dict(os.environ)
            ambiente["SUPABASE_URL"] = _STAGING
            ambiente["SUPABASE_SERVICE_ROLE_KEY"] = chave
            motivo = None
            try:
                for tentativa in range(2):
                    tempo_restante = deadline - time.monotonic() - 120
                    if tempo_restante < 60:
                        motivo = "fim_do_tempo"
                        break
                    try:
                        resultado = subprocess.run(
                            cmd, env=ambiente, check=False,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, errors="replace",
                            timeout=min(1200, int(tempo_restante))
                        )
                        if resultado.returncode == 0:
                            motivo = None
                            break
                        motivo = _classificar_falha(resultado.stdout, resultado.returncode)
                    except subprocess.TimeoutExpired:
                        motivo = "timeout_importacao"
                    if motivo in ("download_overture", "timeout_importacao") and tentativa == 0:
                        print("REPETINDO DOWNLOAD:", cidade + "/" + uf,
                              "| segunda tentativa.", flush=True)
                        time.sleep(5)
                        continue
                    break
            finally:
                ambiente.pop("SUPABASE_SERVICE_ROLE_KEY", None)
                del ambiente
            depois = _contagem(chave, uf, cidade)
            if depois > antes:
                existentes.add(key)
                confirmados += 1
                falhas_consecutivas = 0
                falhas_por_uf[uf] = 0
                print("SALVO:", cidade + "/" + uf, "| empresas:", depois,
                      "| BRASIL:", len(existentes), "/", TOTAL_LOCALIDADES_BRASIL,
                      "| FALTAM:", TOTAL_LOCALIDADES_BRASIL-len(existentes),
                      flush=True)
                continue
            if motivo in ("fim_do_tempo", "teto_do_banco", "autenticacao"):
                print("PARADA:", motivo, "| cidade:", cidade + "/" + uf)
                break
            falhas += 1
            falhas_consecutivas += 1
            falhas_por_uf[uf] = falhas_por_uf.get(uf, 0) + 1
            motivo = motivo or "sem_registros_confirmados"
            try:
                _registrar_pendencia(chave, uf, cidade, motivo)
                pendentes.add(key)
                retry = ("7 dias" if motivo == "sem_dados_overture" else "6 horas")
            except Exception as exc:
                retry = "próxima execução (não foi possível registrar pendência)"
                print("AVISO:", type(exc).__name__, "ao registrar pendência.")
            print("PENDENTE:", cidade + "/" + uf, "| motivo:", motivo,
                  "| nova tentativa:", retry, flush=True)
            if falhas_por_uf[uf] >= MAX_FALHAS_POR_UF:
                ufs_pausadas.add(uf)
                falhas_consecutivas = 0
                print("UF PAUSADA TEMPORARIAMENTE:", uf,
                      "| continuando por outros estados.", flush=True)
        print("SESSÃO ENCERRADA:", confirmados, "NOVOS municípios;",
              falhas, "falhas em recuperação; estados pausados nesta sessão:", len(ufs_pausadas))
        _painel_nacional(existentes, qtd_empresas=_total_empresas(chave),
                         titulo="RESULTADO FINAL DA SESSÃO")
        print("O alvo é nacional, mas tempo do Colab e teto de 250 MiB podem interromper antes.")
        print("Para continuar, execute a célula 4 novamente; ela pula cidades com registros.")
        print("Banco:", round(_espaco(chave)/_MIB, 2), "MiB.")
        print("A amostra de", LIMITE_POR_MUNICIPIO, "registros NÃO é cobertura integral.")
    except HTTPError as exc:
        if exc.code in (401, 403):
            print("AUTENTICAÇÃO RECUSADA PELO SUPABASE (HTTP", exc.code, ").")
            print("Nenhuma nova importação é iniciada se a validação inicial da chave falhar.")
            print("Confira a chave SECRETA no painel da HOMOLOGAÇÃO (prlrxjothrslhswpkoza),")
            print("em Project Settings > API Keys. Use sb_secret_ ou service_role legado.")
            print("Não use anon, publishable, chave de outro projeto ou chave revogada.")
            print("Por segurança, não envie a chave neste chat.")
        else:
            print("ERRO HTTP", exc.code, "durante a consulta à homologação.")
            raise
    except Exception as exc:
        # Somente o TIPO do erro, sem imprimir credenciais ou payloads.
        print("A execução parou antes de finalizar: tipo de erro:", type(exc).__name__)
        print("Os registros já gravados permanecem no Supabase.")
        raise
    finally:
        chave = ""
        print("Credencial encerrada nesta chamada. Desconecte o Colab ao terminar.")

_executar()


## Entendendo o novo registro

**INICIANDO:** a cidade está sendo consultada.
**SALVO:** foram confirmadas empresas dessa cidade no Supabase; ela conta no total brasileiro.
**PENDENTE:** falha ou ausência de estabelecimentos; ela permanece para uma próxima tentativa.
**UF PAUSADA TEMPORARIAMENTE:** falhas repetidas naquele estado; o Colab continua por outros estados.
**SESSÃO ENCERRADA:** resumo com os novos municípios, falhas e armazenamento.

A ausência de uma barra visual no código não desativa os indicadores nativos do ambiente Google Colab.
